# 5.1 訓練迴圈真的正常嗎？

# 第 5 章：訓練與評估形成一個流程

前置：第4章的 logits／shift／loss。程式預設只做結構、梯度與記錄檢查。凡是涉及學到能力的對照，先給可執行配方，之後由讀者在 GPU 上跑；不會把隨機模型的輸出寫成訓練成果。




In [ ]:
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：一次更新的生命週期**

忘記清零會累加；累積梯度時才刻意保留。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/backward.svg")))

**先想一想：**訓練集做不好時，增加新資料一定有用嗎？

先讓模型反覆看極少量正確資料。如果連固定例子都記不住，先查shift、有效labels與梯度，再談泛化。這個測試故意允許背答案。

**把直覺寫成數學：**同一資料的CE可以往0靠近；這是程式診斷，不是最終品質指標。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
from tiny_perceptron.model import TinyLM, ModelConfig, masked_loss

model = TinyLM(ModelConfig(vocab_size=5, width=8))
x, y = torch.tensor([[1, 2, 3]]), torch.tensor([[2, 3, 4]])
loss = masked_loss(model(x)["logits"], y)
loss.backward()
print("初始loss", loss.item(), "embedding梯度", model.embedding.weight.grad.norm().item())
print("讀者訓練入口：python scripts/train.py --task sft --train --steps 200 --batch-size 1")

**如何讀結果：**這格沒有更新權重。真的執行訓練後，畫同一例子loss，若不降就先回查資料。CLI預設合成加法小世界；要精確overfit一例，傳只有一列messages的JSONL。

**只改一件事：**只把labels改成全-100，確認程式直接報錯，而非顯示成功。
